# Attention Head Composition: Tracing Information Flow in Gemma 4

In transformers, attention heads in later layers can **compose** with heads in earlier layers, forming virtual circuits that perform multi-step reasoning. If head A in layer 2 writes information into the residual stream, and head B in layer 5 reads from a similar subspace, then B composes with A.

This notebook traces these composition patterns in **Gemma 4 E4B-it**, following the framework from Elhage et al. (2021), *"A Mathematical Framework for Transformer Circuits"*.

### Three Types of Composition

- **Q-composition**: a later head's Query reads from an earlier head's output
- **K-composition**: a later head's Key reads from an earlier head's output
- **V-composition**: a later head's Value reads from an earlier head's output

### Steps
1. Load model and extract per-head weight matrices (W_Q, W_K, W_V, W_O)
2. Compute OV and QK circuit matrices
3. Measure composition scores between layer pairs
4. Visualize layer-to-layer and head-to-head composition
5. Track information flow through the residual stream
6. Decompose residual stream into attention vs MLP contributions

In [ ]:
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import time
import os
import gc
from collections import defaultdict
from transformers import AutoProcessor, AutoModelForCausalLM

%matplotlib inline

MODEL_ID = "google/gemma-4-E4B-it"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Load Model

Loading Gemma 4 E4B-it with automatic dtype and device mapping. Gemma 4 uses `AutoProcessor` and has 42 decoder layers with Grouped Query Attention (GQA): 8 query heads and 4 KV heads, so each KV head serves 2 query heads.

In [ ]:
print("Loading model and processor...")
t0 = time.time()

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    dtype="auto",
    device_map="auto",
)
model.eval()

load_time = time.time() - t0
print("Model loaded in {:.1f}s".format(load_time))

num_layers = model.config.text_config.num_hidden_layers
hidden_size = model.config.text_config.hidden_size
cfg_head_dim = model.config.text_config.head_dim  # 256

# Gemma 4 hybrid architecture: sliding window layers vs global attention layers
# may have different projection sizes. Detect per-layer dimensions.
layer_info = {}
for i in range(num_layers):
    attn = model.model.language_model.layers[i].self_attn
    q_out = attn.q_proj.weight.shape[0]
    k_out = attn.k_proj.weight.shape[0]
    n_q = q_out // cfg_head_dim
    n_kv = k_out // cfg_head_dim
    layer_info[i] = {"num_q_heads": n_q, "num_kv_heads": n_kv,
                     "head_dim": cfg_head_dim, "q_proj_dim": q_out, "k_proj_dim": k_out}

# Show the distinct layer types
types_seen = {}
for i, info in layer_info.items():
    key = (info["num_q_heads"], info["num_kv_heads"])
    if key not in types_seen:
        types_seen[key] = []
    types_seen[key].append(i)

print("Layers: {}".format(num_layers))
print("Hidden size: {}, Head dim: {}".format(hidden_size, cfg_head_dim))
for (nq, nkv), layers in types_seen.items():
    print("  {} q_heads, {} kv_heads: layers {} ({} layers)".format(
        nq, nkv, layers[:5] if len(layers) > 5 else layers, len(layers)))

## Extract OV and QK Matrices

For each layer and head, we extract the weight matrices that define the attention circuits:

- **OV circuit** (W_V @ W_O): determines *what* information a head writes to the residual stream
- **QK circuit** (W_Q @ W_K^T): determines *which* tokens a head attends to

**Hybrid architecture**: Gemma 4 uses a mix of sliding window and global attention layers. Most layers have 8 query heads and 2 KV heads, but global attention layers (e.g., layer 41) have 16 query heads and 4 KV heads — all with head_dim=256. The code detects per-layer dimensions automatically from weight shapes.

For the OV circuit, we compute per KV-head: W_OV_h = W_V_h @ W_O_h, averaging the O projections of the query heads served by each KV head.

In [ ]:
# Sample layers to keep computation manageable (every 6th layer + last)
sampled_layers = list(range(0, num_layers, 6))
if (num_layers - 1) not in sampled_layers:
    sampled_layers.append(num_layers - 1)
print("Sampled layers: {}".format(sampled_layers))
print("Total sampled: {} out of {}".format(len(sampled_layers), num_layers))

# Extract per-head OV matrices for sampled layers
# Uses per-layer head counts from layer_info to handle hybrid architecture
ov_matrices = {}  # (layer_idx, kv_head_idx) -> (W_V, W_O) factored
q_matrices = {}   # (layer_idx, q_head_idx) -> W_Q_h (head_dim x hidden_size)
k_matrices = {}   # (layer_idx, kv_head_idx) -> W_K_h (head_dim x hidden_size)

print("Extracting weight matrices...")
t0 = time.time()

for li in sampled_layers:
    info = layer_info[li]
    n_q = info["num_q_heads"]
    n_kv = info["num_kv_heads"]
    hd = info["head_dim"]
    kv_group = n_q // n_kv

    layer = model.model.language_model.layers[li]
    attn = layer.self_attn

    W_Q = attn.q_proj.weight.detach().cpu().float()  # (n_q * hd, hidden_size)
    W_K = attn.k_proj.weight.detach().cpu().float()  # (n_kv * hd, hidden_size)
    W_V = attn.v_proj.weight.detach().cpu().float()  # (n_kv * hd, hidden_size)
    W_O = attn.o_proj.weight.detach().cpu().float()  # (hidden_size, n_q * hd)

    W_Q_heads = W_Q.view(n_q, hd, hidden_size)
    W_K_heads = W_K.view(n_kv, hd, hidden_size)
    W_V_heads = W_V.view(n_kv, hd, hidden_size)
    W_O_heads = W_O.view(hidden_size, n_q, hd)

    for kv_h in range(n_kv):
        W_V_h = W_V_heads[kv_h]
        q_start = kv_h * kv_group
        q_end = q_start + kv_group
        W_O_h = W_O_heads[:, q_start:q_end, :].mean(dim=1)  # (hidden_size, hd)
        ov_matrices[(li, kv_h)] = (W_V_h, W_O_h)

    for qh in range(n_q):
        q_matrices[(li, qh)] = W_Q_heads[qh]
    for kv_h in range(n_kv):
        k_matrices[(li, kv_h)] = W_K_heads[kv_h]

    print("  Layer {}: {} q_heads, {} kv_heads".format(li, n_q, n_kv))

print("Extracted in {:.1f}s".format(time.time() - t0))
print("OV matrices: {}, Q matrices: {}, K matrices: {}".format(
    len(ov_matrices), len(q_matrices), len(k_matrices)))

## Composition Scores

We measure how much later heads compose with earlier heads using a simplified composition score.

For V-composition between head A (layer i) and head B (layer j, j > i):
- Head A writes to the residual stream via its OV circuit
- Head B reads from the residual stream via its QK circuit
- Composition strength = how much B's input subspace overlaps with A's output subspace

**Simplified metric**: We compute the mean absolute cosine similarity between the output directions of layer i's OV circuit (columns of W_O) and the input directions of layer j's QK circuit (rows of W_Q and W_K). This gives a proxy for composition strength without materializing the full hidden_size x hidden_size matrices.

In [ ]:
def compute_composition_score(ov_factored, q_or_k_matrix):
    """Compute composition score between an OV circuit and a Q/K matrix.

    ov_factored: (W_V, W_O) where W_V is (head_dim, hidden_size), W_O is (hidden_size, head_dim)
    q_or_k_matrix: (head_dim, hidden_size)

    Score = ||q_or_k @ W_O||_F / (||q_or_k||_F * ||W_O||_F)
    """
    W_V, W_O = ov_factored
    product = q_or_k_matrix @ W_O  # (head_dim_q, head_dim_o)
    score = torch.norm(product, p="fro") / (torch.norm(q_or_k_matrix, p="fro") * torch.norm(W_O, p="fro") + 1e-8)
    return score.item()


# Compute layer-to-layer composition scores
n_sampled = len(sampled_layers)
layer_composition = np.zeros((n_sampled, n_sampled))

print("Computing layer-to-layer composition scores...")
t0 = time.time()

for i_idx, li in enumerate(sampled_layers):
    n_kv_early = layer_info[li]["num_kv_heads"]
    for j_idx, lj in enumerate(sampled_layers):
        if lj <= li:
            continue
        n_q_late = layer_info[lj]["num_q_heads"]
        n_kv_late = layer_info[lj]["num_kv_heads"]

        total_score = 0.0
        count = 0
        for kv_h_early in range(n_kv_early):
            ov_early = ov_matrices[(li, kv_h_early)]
            # Q-composition
            for qh_late in range(n_q_late):
                q_late = q_matrices[(lj, qh_late)]
                total_score += compute_composition_score(ov_early, q_late)
                count += 1
            # K-composition
            for kv_h_late in range(n_kv_late):
                k_late = k_matrices[(lj, kv_h_late)]
                total_score += compute_composition_score(ov_early, k_late)
                count += 1

        layer_composition[i_idx, j_idx] = total_score / count if count > 0 else 0.0

print("Computed in {:.1f}s".format(time.time() - t0))
print("Max composition score: {:.4f}".format(layer_composition.max()))
print("Mean composition score (upper triangle): {:.4f}".format(
    layer_composition[np.triu_indices(n_sampled, k=1)].mean()))

## Layer-to-Layer Composition Heatmap

This heatmap shows the total composition strength between each pair of sampled layers. Only the upper triangle is meaningful (later layers composing with earlier layers). Bright cells indicate strong composition, meaning the later layer's QK circuit is aligned with the earlier layer's OV output subspace.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Mask lower triangle
mask = np.tril(np.ones_like(layer_composition, dtype=bool))
masked_comp = np.ma.masked_where(mask, layer_composition)

im = ax.imshow(masked_comp, cmap="YlOrRd", aspect="equal", interpolation="nearest")
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Composition Score", fontsize=12)

ax.set_xticks(range(n_sampled))
ax.set_yticks(range(n_sampled))
ax.set_xticklabels(["L{}".format(l) for l in sampled_layers], fontsize=9)
ax.set_yticklabels(["L{}".format(l) for l in sampled_layers], fontsize=9)
ax.set_xlabel("Later Layer (reads)", fontsize=12)
ax.set_ylabel("Earlier Layer (writes)", fontsize=12)
ax.set_title("Gemma 4 \u2014 Attention Head Composition Strength", fontsize=14, fontweight="bold")

# Annotate cells
for i in range(n_sampled):
    for j in range(n_sampled):
        if j > i:
            val = layer_composition[i, j]
            color = "white" if val > masked_comp.max() * 0.6 else "black"
            ax.text(j, i, "{:.3f}".format(val), ha="center", va="center",
                    fontsize=7, color=color)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_17_composition_heatmap.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## Head-Level Composition

Now we zoom in to see which specific heads compose with each other across the sampled layers. Each cell shows the composition score between a specific (layer, head) pair in the earlier layer and a specific (layer, head) pair in the later layer.

In [ ]:
# Build head-level composition matrix across sampled layers
# Use KV heads for both sides. Variable KV head counts per layer.
# Build a flat index: (layer_idx, kv_head) pairs
head_labels = []
for i_idx, li in enumerate(sampled_layers):
    n_kv = layer_info[li]["num_kv_heads"]
    for kv_h in range(n_kv):
        head_labels.append((li, kv_h))

n_total_heads = len(head_labels)
head_composition = np.zeros((n_total_heads, n_total_heads))

print("Computing head-level composition ({} x {} matrix)...".format(n_total_heads, n_total_heads))
t0 = time.time()

for row, (li, kv_h_early) in enumerate(head_labels):
    ov_early = ov_matrices[(li, kv_h_early)]
    for col, (lj, kv_h_late) in enumerate(head_labels):
        if lj <= li:
            continue
        k_late = k_matrices[(lj, kv_h_late)]
        score = compute_composition_score(ov_early, k_late)
        head_composition[row, col] = score

print("Computed in {:.1f}s".format(time.time() - t0))

# Plot
fig, ax = plt.subplots(figsize=(14, 12))

mask = np.tril(np.ones_like(head_composition, dtype=bool))
masked_hc = np.ma.masked_where(mask, head_composition)

im = ax.imshow(masked_hc, cmap="inferno", aspect="equal", interpolation="nearest")
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("K-Composition Score", fontsize=12)

# Label at layer boundaries
tick_positions = []
tick_labels_list = []
offset = 0
for i_idx, li in enumerate(sampled_layers):
    n_kv = layer_info[li]["num_kv_heads"]
    tick_positions.append(offset + n_kv // 2)
    tick_labels_list.append("L{}".format(li))
    offset += n_kv

ax.set_xticks(tick_positions)
ax.set_yticks(tick_positions)
ax.set_xticklabels(tick_labels_list, fontsize=9)
ax.set_yticklabels(tick_labels_list, fontsize=9)

# Draw layer boundary lines
offset = 0
for i_idx, li in enumerate(sampled_layers[:-1]):
    offset += layer_info[li]["num_kv_heads"]
    pos = offset - 0.5
    ax.axhline(y=pos, color="white", linewidth=0.5, alpha=0.5)
    ax.axvline(x=pos, color="white", linewidth=0.5, alpha=0.5)

ax.set_xlabel("Later Layer Heads (reads)", fontsize=12)
ax.set_ylabel("Earlier Layer Heads (writes)", fontsize=12)
ax.set_title("Gemma 4 — Head-Level K-Composition Across Layers", fontsize=14, fontweight="bold")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_18_head_composition.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## Information Flow Diagram

We hook every layer's attention module and measure how much each head contributes to the residual stream for a specific input. This shows the magnitude of each head's output, giving us a picture of information flow through the network.

We use the prompt *"The Eiffel Tower is located in"* and track how each attention head's contribution norm varies across all 42 layers.

In [ ]:
PROMPT = "The Eiffel Tower is located in"
print("Prompt: \"{}\"".format(PROMPT))

# Storage for per-layer attention output norms (total and per-head)
head_contributions = {}  # layer_idx -> array of per-head norms

def make_oproj_input_hook(layer_idx):
    """Hook on o_proj to capture its INPUT (pre-projection per-head outputs)."""
    def hook(module, input, output):
        inp = input[0].detach().cpu().float()
        if inp.dim() == 2:
            inp = inp.unsqueeze(0)
        last_token = inp[0, -1, :]  # (n_q * head_dim,)
        n_q = layer_info[layer_idx]["num_q_heads"]
        hd = layer_info[layer_idx]["head_dim"]
        head_contribs = last_token.view(n_q, hd)
        norms = torch.norm(head_contribs, dim=1)  # (n_q,)
        head_contributions[layer_idx] = norms.numpy()
    return hook

handles = []
for i in range(num_layers):
    layer = model.model.language_model.layers[i]
    h = layer.self_attn.o_proj.register_forward_hook(make_oproj_input_hook(i))
    handles.append(h)

inputs = processor(text=PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    _ = model(**inputs)

for h in handles:
    h.remove()

print("Captured head contributions for {} layers".format(len(head_contributions)))

In [ ]:
# Build matrix: (num_layers, max_q_heads) — pad layers with fewer heads
max_q_heads = max(info["num_q_heads"] for info in layer_info.values())
flow_matrix = np.zeros((num_layers, max_q_heads))
for li in range(num_layers):
    if li in head_contributions:
        n_q = len(head_contributions[li])
        flow_matrix[li, :n_q] = head_contributions[li]

fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(num_layers)
colors = plt.cm.Set2(np.linspace(0, 1, max_q_heads))
bottom = np.zeros(num_layers)

for h in range(max_q_heads):
    ax.bar(x, flow_matrix[:, h], bottom=bottom, color=colors[h],
           label="Head {}".format(h), alpha=0.85, width=0.9)
    bottom += flow_matrix[:, h]

ax.set_xlabel("Layer", fontsize=12)
ax.set_ylabel("Head Contribution Norm", fontsize=12)
ax.set_title("Gemma 4 — Per-Head Information Flow: \"{}\"".format(PROMPT),
             fontsize=14, fontweight="bold")
ax.legend(loc="upper left", ncol=4, fontsize=8, framealpha=0.8)
ax.set_xlim(-0.5, num_layers - 0.5)
ax.grid(axis="y", alpha=0.3)

# Mark global attention layer(s) with different head count
for li, info in layer_info.items():
    if info["num_q_heads"] != layer_info[0]["num_q_heads"]:
        ax.axvline(x=li, color="red", linestyle="--", alpha=0.5, linewidth=1)
        ax.text(li, ax.get_ylim()[1] * 0.95, "global", fontsize=7,
                ha="center", color="red", alpha=0.7)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_19_information_flow.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## Residual Stream Decomposition

For each layer, the residual stream update can be decomposed into two additive components:
- **Attention contribution**: output of the self-attention sub-layer
- **MLP contribution**: output of the MLP sub-layer

We hook both `self_attn` and `mlp` in every layer to capture the norm of each contribution, then visualize how the relative importance of attention vs MLP changes across all 42 layers.

In [ ]:
attn_norms = {}
mlp_norms = {}

def make_component_hook(storage, layer_idx):
    def hook(module, input, output):
        if isinstance(output, tuple):
            tensor = output[0]
        else:
            tensor = output
        if isinstance(tensor, torch.Tensor):
            t = tensor.detach().cpu().float()
            if t.dim() == 2:
                t = t.unsqueeze(0)
            # Norm across the entire output
            storage[layer_idx] = t.norm().item()
    return hook

# Attach hooks to both attention and MLP
handles = []
for i in range(num_layers):
    layer = model.model.language_model.layers[i]
    h1 = layer.self_attn.register_forward_hook(make_component_hook(attn_norms, i))
    h2 = layer.mlp.register_forward_hook(make_component_hook(mlp_norms, i))
    handles.extend([h1, h2])

# Forward pass
with torch.no_grad():
    _ = model(**inputs)

for h in handles:
    h.remove()

print("Captured attention norms for {} layers".format(len(attn_norms)))
print("Captured MLP norms for {} layers".format(len(mlp_norms)))

In [ ]:
layers_x = np.arange(num_layers)
attn_vals = np.array([attn_norms.get(i, 0.0) for i in range(num_layers)])
mlp_vals = np.array([mlp_norms.get(i, 0.0) for i in range(num_layers)])

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]})

# Stacked area chart
ax = axes[0]
ax.fill_between(layers_x, 0, attn_vals, alpha=0.7, color="royalblue", label="Attention")
ax.fill_between(layers_x, attn_vals, attn_vals + mlp_vals, alpha=0.7, color="tomato", label="MLP")
ax.set_xlabel("Layer", fontsize=12)
ax.set_ylabel("Output Norm", fontsize=12)
ax.set_title("Gemma 4 \u2014 Residual Stream Decomposition: Attention vs MLP",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="upper left")
ax.set_xlim(0, num_layers - 1)
ax.grid(alpha=0.3)

# Ratio chart
ax2 = axes[1]
total = attn_vals + mlp_vals + 1e-8
attn_frac = attn_vals / total
mlp_frac = mlp_vals / total
ax2.fill_between(layers_x, 0, attn_frac, alpha=0.7, color="royalblue", label="Attention")
ax2.fill_between(layers_x, attn_frac, 1.0, alpha=0.7, color="tomato", label="MLP")
ax2.set_xlabel("Layer", fontsize=12)
ax2.set_ylabel("Fraction", fontsize=12)
ax2.set_title("Relative Contribution (Attention vs MLP)", fontsize=12)
ax2.set_xlim(0, num_layers - 1)
ax2.set_ylim(0, 1)
ax2.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "g4_20_residual_decomposition.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## Summary

This notebook traced information flow and attention head composition in Gemma 4 E4B-it, applying the framework from Elhage et al. (2021).

In [ ]:
# Summarize key findings
print("=" * 70)
print("ATTENTION HEAD COMPOSITION ANALYSIS — GEMMA 4 E4B-it")
print("=" * 70)
print()

# Find strongest composition pairs
print("1. LAYER-TO-LAYER COMPOSITION")
print("-" * 40)
top_pairs = []
for i_idx in range(n_sampled):
    for j_idx in range(i_idx + 1, n_sampled):
        top_pairs.append((sampled_layers[i_idx], sampled_layers[j_idx],
                          layer_composition[i_idx, j_idx]))
top_pairs.sort(key=lambda x: x[2], reverse=True)

print("Top 5 composing layer pairs:")
for li, lj, score in top_pairs[:5]:
    print("  Layer {} -> Layer {}: {:.4f}".format(li, lj, score))

print()
print("2. INFORMATION FLOW")
print("-" * 40)
total_per_layer = flow_matrix.sum(axis=1)
top_flow_layers = np.argsort(total_per_layer)[::-1][:5]
print("Layers with highest total head contribution:")
for li in top_flow_layers:
    n_q = layer_info[li]["num_q_heads"]
    dominant_head = np.argmax(flow_matrix[li, :n_q])
    print("  Layer {}: total norm = {:.2f}, dominant head = {} ({} q_heads)".format(
        li, total_per_layer[li], dominant_head, n_q))

print()
print("3. RESIDUAL STREAM DECOMPOSITION")
print("-" * 40)
mean_attn_frac = attn_frac.mean()
print("Average attention fraction: {:.1f}%".format(mean_attn_frac * 100))
print("Average MLP fraction: {:.1f}%".format((1 - mean_attn_frac) * 100))

attn_dominant = np.where(attn_frac > 0.5)[0]
mlp_dominant = np.where(mlp_frac > 0.5)[0]
if len(attn_dominant) > 0:
    print("Attention-dominant layers: {} (out of {})".format(len(attn_dominant), num_layers))
if len(mlp_dominant) > 0:
    print("MLP-dominant layers: {} (out of {})".format(len(mlp_dominant), num_layers))

print()
print("4. KEY FINDINGS")
print("-" * 40)
print("- Gemma 4 uses HYBRID attention with variable head counts:")
for (nq, nkv), layers in types_seen.items():
    print("    {} q_heads, {} kv_heads: {} layers".format(nq, nkv, len(layers)))
print("- Composition scores reveal virtual circuits between layer pairs")
print("- The information flow diagram shows per-head contribution norms")
print("  across all {} layers".format(num_layers))
print("- Residual stream decomposition shows attention vs MLP roles")
print()
print("Charts saved to: {}".format(OUTPUT_DIR))
print("  - g4_17_composition_heatmap.png")
print("  - g4_18_head_composition.png")
print("  - g4_19_information_flow.png")
print("  - g4_20_residual_decomposition.png")